# 08 · Runnable Integration & Security Tests

Exercises data isolation, safe ZIP extraction, document parsing, cross-domain retrieval, unsupported candidate abstention, quantitative analysis, and the live FastAPI ASGI app. No third-party network access required.

In [ ]:
%run 07_fastapi_integration.ipynb

In [ ]:
%run 05_evaluation.ipynb
import asyncio
os.environ['GROUP2_DEV_MODE']='true'
from httpx import ASGITransport, AsyncClient

async def run_api_checks():
    seed_synthetic()
    other_id='88888888-8888-4888-8888-888888888888'
    async with AsyncClient(transport=ASGITransport(app=app),base_url='http://test') as client:
        health_response=await client.get('/health');assert health_response.status_code==200
        response=await client.post(f'/api/v1/projects/{PROJECT_ID}/knowledge/search',
                                   json={'question':'DemoRouter-X revision B optical module','domain':'technology'})
        assert response.status_code==200 and response.json()['evidence']
        response=await client.post(f'/api/v1/projects/{other_id}/knowledge/search',
                                   json={'question':'DemoRouter-X revision B optical module','domain':'technology'})
        assert response.status_code==200 and not response.json()['evidence'], 'Project boundary violated'
        response=await client.post(f'/api/v1/projects/{PROJECT_ID}/recommendations/rank',json={
            'query':'optical module','domain':'technology','device_model':'DemoRouter-X revision B',
            'candidates':[{'candidate_id':'approved','name':'DemoSFP-10G','kind':'replacement_part',
                           'compatible_models':['DemoRouter-X revision B']}]})
        assert response.status_code==200 and response.json()['recommendations']
        response=await client.post('/api/v1/projects/not-a-uuid/knowledge/search',json={'question':'valid query'})
        assert response.status_code==422
        response=await client.post(f'/api/v1/projects/{PROJECT_ID}/knowledge/search',
            json={'project_id':other_id,'question':'DemoRouter-X revision B','domain':'technology'})
        assert response.status_code==200 and response.json()['evidence'], 'Payload must not override authorized project scope'
        response=await client.post(f'/api/v1/projects/{PROJECT_ID}/recommendations/rank',
            json={'project_id':other_id,'query':'router','domain':'technology','device_model':'DemoRouter-X revision B',
             'candidates':[{'candidate_id':'sfp','name':'DemoSFP-10G','kind':'replacement_part',
             'compatible_models':['DemoRouter-X revision B']}]})
        assert response.status_code==200 and response.json()['recommendations'], 'Recommender must enforce project path scope'
        response=await client.post(f'/api/v1/projects/{PROJECT_ID}/research/tasks',json={
            'domain':'csr','question':'cold storage reducing post-harvest spoilage',
            'source_mode':'synthetic_fixture','top_k':3})
        assert response.status_code==200 and response.json()['findings']
        task_id=response.json()['research_task_id']
        readback=await client.get(f'/api/v1/projects/{PROJECT_ID}/research/tasks/{task_id}')
        assert readback.status_code==200 and readback.json()['research_task_id']==task_id
        data=await client.post(f'/api/v1/projects/{PROJECT_ID}/analytics/financial',json={
            'rows':[{'year':2024,'revenue':100,'cost':70},{'year':2025,'revenue':125,'cost':80}],
            'source_name':'SYNTHETIC API test'})
        assert data.status_code==200 and data.json()['cagr_pct']==25.0
    return True

# This notebook is designed to run with Jupyter/IPython's running event loop.
await run_api_checks()
print('PASS: API, project isolation, schema validation, and tool output')

In [ ]:
from tempfile import NamedTemporaryFile
import zipfile as _zip
seed_synthetic()
assert all(test_recommendation_safety().values())
try:
    ingest_blob(PROJECT_ID,'technology','bad.pdf',b'not-a-pdf')
    raise AssertionError('Invalid file accepted')
except ValueError:pass
with NamedTemporaryFile(suffix='.zip') as tmp:
    with _zip.ZipFile(tmp.name,'w') as z:z.writestr('../outside.txt','bad')
    try:ingest_archive(tmp.name,'technology')
    except ValueError:pass
    else:raise AssertionError('Unsafe ZIP path accepted')

finance=financial_metrics([{'year':2023,'revenue':100,'cost':70},
                           {'year':2024,'revenue':110,'cost':75}],PROJECT_ID)
assert round(finance['cagr_pct'],2)==10.0
assert round(finance['latest_margin_pct'],2)==round(100*35/110,2)
assert len(hybrid_search(PROJECT_ID,'post harvest solar cold storage','csr',3))>0
assert hybrid_search('88888888-8888-4888-8888-888888888888','SampleTray-200','clinical')==[]
metrics=evaluate_fixture()
assert metrics['citation_validity_rate']==1
assert metrics['Recall@K']>=0.8
print('PASS: document security, ZIP safety, financial calculations, domain isolation, retrieval evaluation')
print(json.dumps({k:v for k,v in metrics.items() if k!='details'},indent=2))